In [1]:
# Copyright 2026 NVIDIA Corporation
# SPDX-License-Identifier: Apache-2.0

In [2]:
import argparse
import json
import shutil
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from safetensors import safe_open
from safetensors.torch import save_file
from transformers import AutoTokenizer

/sadata/nikkothari/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Training 2xh100 6 days # 2M datapoints # 1200 ISL Average / 10 QPS (A100) => h200/h100
base = Path("/sadata/nikkothari/cache/hf/hub/models--unsloth--gemma-3-4b-it/snapshots/bf46152c47f5dd20b896357cb51abc4c03b8ee8c/")
labels = 7000 # 6948 
out = ".cache/models/gemma3-4b-cls7000"

In [4]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained(base)
config.architectures = ["Gemma3ForSequenceClassification"]

config.id2label = {str(i): f"{i:04d}" for i in range(labels)}
config.label2id = {name: i for i, name in enumerate(config.id2label.values())}
config.text_config.pad_token_id = config.pad_token_id

In [8]:
config.architectures, config.num_labels

(['Gemma3ForSequenceClassification'], 7000)

In [7]:
import torch
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(base, config=config, dtype=torch.bfloat16)
model

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████| 883/883 [00:00<00:00, 7779.65it/s]
[transformers] Gemma3ForSequenceClassification LOAD REPORT from: /sadata/nikkothari/cache/hf/hub/models--unsloth--gemma-3-4b-it/snapshots/bf46152c47f5dd20b896357cb51abc4c03b8ee8c
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Gemma3ForSequenceClassification(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (embeddings): SiglipVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(4096, 1152)
      )
      (encoder): SiglipEncoder(
        (layers): ModuleList(
          (0-26): 27 x SiglipEncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
            (self_attn): SiglipAttention(
              (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
            (mlp): SiglipMLP(
        

In [ ]:
from transformers import AutoProcessor

out = "/sadata/nikkothari/walmart/gemma3-4b-optimization-vllm/.cache/models/gemma3-4b-cls7000"
model.save_pretrained(out)
AutoProcessor.from_pretrained(base).save_pretrained(out)

# vLLM Parity

In [1]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("/sadata/nikkothari/walmart/gemma3-4b-optimization-vllm/.cache/models/gemma3-4b-cls7000")

/sadata/nikkothari/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████| 884/884 [00:04<00:00, 210.04it/s]


In [2]:
model.to("cuda:3")

Gemma3ForSequenceClassification(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (embeddings): SiglipVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(4096, 1152)
      )
      (encoder): SiglipEncoder(
        (layers): ModuleList(
          (0-26): 27 x SiglipEncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
            (self_attn): SiglipAttention(
              (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
            (mlp): SiglipMLP(
        

In [5]:
import base64, json, urllib.request
from pathlib import Path
from PIL import Image


image_path = Path("/sadata/nikkothari/walmart/gemma3-4b-optimization/data/images/cigarette_box-good-005_001.jpg")
instruction = ("You are a product taxonomy classifier for an online retail catalog. Look at the product photo and "
               "read the listing that follows it, then assign the product to the single most specific category of "
               "the catalog taxonomy. Consider the packaging, the brand, the product form, the ingredients or "
               "materials and the intended use. Answer with the category name only.")
text = "Title: Classic cigarette box, 20 king size filter cigarettes, hard pack."
uri = "data:image/jpeg;base64," + base64.b64encode(image_path.read_bytes()).decode()

vllm_body = {"messages": [{"role": "user", "content": [
                {"type": "text", "text": instruction},
                {"type": "image_url", "image_url": {"url": uri}},
                {"type": "text", "text": text}]}],
             "add_generation_prompt": True, "use_activation": False}

hf_messages = [{"role": "user", "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": Image.open(image_path).convert("RGB")},
                {"type": "text", "text": text}]}]

In [7]:
import torch
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("/sadata/nikkothari/walmart/gemma3-4b-optimization-vllm/.cache/models/gemma3-4b-cls7000")
inputs = processor.apply_chat_template(hf_messages, add_generation_prompt=True, tokenize=True,
                                       return_dict=True, return_tensors="pt").to(model.device)
inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)
model.eval()
with torch.inference_mode():
    hf_logits = model(**inputs).logits[0].float().cpu()

In [11]:
hf_logits.shape

torch.Size([7000])

In [12]:
hf_logits

tensor([  4.9062,   4.5625,   9.7500,  ..., -22.8750, -19.1250,  -1.7109])

In [9]:
req = urllib.request.Request("http://127.0.0.1:8100/classify", data=json.dumps(vllm_body).encode(),
                             headers={"Content-Type": "application/json"})
with urllib.request.urlopen(req) as r:
    vllm_answer = json.load(r)
vllm_logits = torch.tensor(vllm_answer["data"][0]["probs"])

In [16]:
names = model.config.id2label

print("prompt tokens  HF:", inputs["input_ids"].shape[1], " vLLM:", vllm_answer["usage"]["prompt_tokens"])
print("cosine sim    :", torch.nn.functional.cosine_similarity(hf_logits, vllm_logits, dim=0).item())
print("max |diff|    :", (hf_logits - vllm_logits).abs().max().item())
print("top-1 agree   :", hf_logits.argmax().item() == vllm_logits.argmax().item())
for name, l in [("HF", hf_logits), ("vLLM", vllm_logits)]:
    v, i = l.topk(5)
    print(f"{name:5}", [(names[k], round(x, 3)) for k, x in zip(i.tolist(), v.tolist())])

prompt tokens  HF: 353  vLLM: 353
cosine sim    : 0.9987374544143677
max |diff|    : 2.8026914596557617
top-1 agree   : True
HF    [('6454_smoke', 19.875), ('2965_finished', 17.125), ('2382_proceed', 17.125), ('1278_contr', 16.5), ('0235_case', 16.5)]
vLLM  [('6454_smoke', 18.368), ('0235_case', 18.123), ('2382_proceed', 17.829), ('1278_contr', 16.226), ('2965_finished', 16.173)]
